# Interaction strength calculation

## Imports

In [5]:
# import standard python libraries
import matplotlib as mpl
%matplotlib inline
mpl.rcParams['figure.dpi'] = 96
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# import libraries for biological data analysis
from coolpuppy import coolpup
from plotpuppy import plotpup
import cooler
import bioframe
import cooltools
from cooltools import expected_cis
from cooltools.lib import plotting
from cooltools.api.snipping import ObsExpSnipper

import bbi

## Inputs

First, get the loops and mcools to analyse, set up variables, etc

In [2]:
#mcool resolution to read in bp
resolution = 500
#List of mcool paths as strings
conditions = ["dTAG_RING1A_B_dTAG_SUZ12_UNT_r1_merged.merged.nodups.transfiltered.50.mcool", "dTAG_RING1A_B_dTAG_SUZ12_dTAG_r1_merged.merged.nodups.transfiltered.50.mcool"]
conditionNames = ["dTAG-RING1AB_dTAG-SUZ12_UNT_r1", "dTAG-RING1AB_dTAG-SUZ12_dTAG_r1"]


#List of loop types as strings
loopTypesNames = ["PcG"]
#List of loop file locations (bedpe)
loopFiles = ["GSE119618_RING1B_SUZ12.pairedpeaks.mm39.bedpe"]
#Specify the RCMC regions of the mcools to look at (format: chromosome (string), start (number), end (number), name of region (string))
regions = pd.read_csv("capture_loci.bed", sep='\t', names=['chrom', 'start', 'end', 'name'], header=None)
#Set save directory
saveDir = '/path/to/save/directory/'

#Set the size of the area flanking the interacting regions (in case you are not sure of exact positions)
pad = 0


## Get loops

In [2]:
#Function to import loop files
def getLoops(loopFiles, loopTypesNames):
    loopTypes = []
    colNames = ['chrom1', 'start1', 'end1', 'chrom2', 'start2', 'end2']
    for file in loopFiles:
        temploops = pd.read_csv(file, sep='\t', names=colNames, header=None)
        loopTypes.append(temploops)
    loopDict = dict(zip(loopTypesNames, loopTypes))
    return loopDict

In [7]:
loopDict = getLoops(loopFiles, loopTypesNames)

## Compute expected

In [8]:
#Function to compute expected matrices for all conditions
def getExpected(conditions, regions, resolution):
    expecteds = {}   
    for condition in conditions:
        print(condition)
        #Get cooler
        clr = cooler.Cooler(condition+'::/resolutions/'+str(resolution))
        expected = expected_cis(
            clr,
            view_df=regions.sort_values('chrom'),
            nproc=32)
        expecteds.update({condition:expected})
    return expecteds

In [ ]:
#Get expected matrices for all coolers
expecteds = getExpected(conditions, regions, resolution)


## Functions for getting loop info and getting enrichments

In [12]:
#Function to get loop info
def getLoopInfo(loops, loopcount, pad):
    #Get loop position
    #Probably need to pad here
    uppos_1 = loops.iloc[loopcount][1] - pad
    uppos_2 = loops.iloc[loopcount][2] + pad
    downpos_1 = loops.iloc[loopcount][4] - pad
    downpos_2 = loops.iloc[loopcount][5] + pad
    looploc = (uppos_1, uppos_2, downpos_1, downpos_2)
    #Then repeat this process for a box upstream and downstream
    #I want these to not overlap at all in x or y
    #Generate an upstream box
    #Expand the rectangle from the upper left corner to make an imaginary square, then find the identically-sized squares touching the corners
    dim1 = uppos_2 - uppos_1
    dim2 = downpos_2 - downpos_1
    maxlength = max(dim1, dim2)
    
    #Determine the region to use
    region = regions[(regions['chrom'] == loops.iloc[loopcount][0]) & (regions['start'] <= loops.iloc[loopcount][1]) & (regions['end'] >= loops.iloc[loopcount][1])].iloc[0][3]
    print(region)
    
    #Then get the coordinates that would belong to the upstream position
    uploc = (uppos_1 - maxlength, uppos_1 - maxlength + dim1, downpos_1 - maxlength, downpos_1 - maxlength + dim2)
    #And do the same for downstream
    downloc = (uppos_1 + maxlength, uppos_1 + maxlength + dim1, downpos_1 + maxlength, downpos_1 + maxlength + dim2)
    
    #Finally, need to check that the locations of the upstream and downstream boxes are inside the region, 
    # because if they are outside, they should be skipped
    regstart = regions[(regions['name'] == region)].iloc[0][1]
    regend = regions[(regions['name'] == region)].iloc[0][2]
    if uploc[0] <= regstart:
        upcheck = 1
    else:
        upcheck = 0
    if downloc[3] >= regend:
        downcheck = 1
    else:
        downcheck = 0

    return looploc, uploc, downloc, region, upcheck, downcheck

In [13]:
#Get obs/exp read count for box
def getSum(snipper, region, location):
    sniparray = snipper.snip(matrix=snipper.select(region, region), region1=region, region2=region, tup=location)
    boxsum = np.nansum(sniparray)
    return boxsum

In [14]:
#Enrichment function for rectangles
def getRectangleEnrich(condition, expecteds, resolution, looploc, uploc, downloc, loopcount, region, upcheck, downcheck):
    clr = cooler.Cooler(condition+'::/resolutions/'+str(resolution))
    expected = expecteds[condition]
    #Use the snipper to fetch the relevant array
    snipper = cooltools.api.snipping.ObsExpSnipper(clr=clr, expected=expected, view_df=regions)
    roi_sum = getSum(snipper, region, looploc)
    #Get box sum for upstream box
    if upcheck:
        print("Skipping upstream box")
        upsum = 0
    else:
        upsum = getSum(snipper, region, uploc)
    #Get box sum for downstream box
    if downcheck:
        print("Skipping downstream box")
        downsum = 0
    else:
        downsum = getSum(snipper, region, downloc)
    #If there's nothing in either normalisation box, either because they're blank or at the edges of the region, 
    #don't bother calculating anything.
    if upsum + downsum == 0:
        print("No comparisons, skipping loop")
        enrichment = np.nan
    else:
        #Also, if the region box had nothing in it, don't bother calculating either.
        if roi_sum == 0:
            print("No loop, skipping")
            enrichment = np.nan
        elif upcheck:
            print("Value without upstream box")
            enrichment = roi_sum/downsum
        elif downcheck:
            print("Value without downstream box")
            enrichment = roi_sum/upsum
        else:
            enrichment = roi_sum/((upsum + downsum)/2)
    return enrichment

## Enrichment calculation loop

In [ ]:
enrichDict = dict.fromkeys(loopTypesNames)

for loopsName in loopDict:
    print(loopsName)
    loops = loopDict[loopsName]
    loops[conditionNames] = np.nan
    for i in range(len(loops)):
        print(f"Loop {i}:")
        looploc, uploc, downloc, region, upcheck, downcheck = getLoopInfo(loops, i, pad)
        for condition, conditionName in zip(conditions, conditionNames):
            enrichment = getRectangleEnrich(condition, expecteds, resolution, looploc, uploc, downloc, i, region, upcheck, downcheck)
            print(f"{conditionName} {enrichment}")
            loops.at[i, conditionName] = enrichment
    enrichDict[loopsName] = loops

In [16]:
conditionNamesCombined = "_".join(conditionNames)

for name, df in enrichDict.items():
    df.to_csv(saveDir + name + '_' + 'loopstrength.bedpe', sep = '\t', index = False, header = True)